# Module 7 – Practical Task 1: Customer Segmentation
## Customer Personality Analysis using Clustering

**Objective:** Segment customers based on their personality, spending behaviour, and demographics using:
1. K-Means Clustering
2. Hierarchical Clustering
3. Comparison and Conclusion


## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## Step 2: Create / Load Dataset

We create a synthetic Customer Personality dataset with realistic features.

In [ ]:
np.random.seed(42)
N = 800

# Simulate 3 customer groups
age = np.concatenate([
    np.random.normal(27, 5, int(N*0.30)),   # Young
    np.random.normal(42, 6, int(N*0.40)),   # Middle-aged
    np.random.normal(58, 7, int(N*0.30))    # Senior
])[:N]

income = np.concatenate([
    np.random.normal(32000, 7000, int(N*0.30)),
    np.random.normal(62000, 12000, int(N*0.40)),
    np.random.normal(105000, 22000, int(N*0.30))
])[:N]

spending_score = np.concatenate([
    np.random.normal(30, 8, int(N*0.30)),
    np.random.normal(55, 10, int(N*0.40)),
    np.random.normal(80, 12, int(N*0.30))
])[:N]

loyalty = np.concatenate([
    np.random.choice([1,2], int(N*0.30)),
    np.random.choice([2,3,4], int(N*0.40)),
    np.random.choice([4,5], int(N*0.30))
])[:N]

recency = np.concatenate([
    np.random.randint(40, 180, int(N*0.30)),
    np.random.randint(10, 60,  int(N*0.40)),
    np.random.randint(1,  25,  int(N*0.30))
])[:N]

frequency = np.concatenate([
    np.random.randint(1, 4,  int(N*0.30)),
    np.random.randint(3, 10, int(N*0.40)),
    np.random.randint(8, 20, int(N*0.30))
])[:N]

gender    = np.random.choice(['Male','Female'], N)
education = np.random.choice(['High School','Graduate','Post-Graduate'], N, p=[0.25,0.45,0.30])

df = pd.DataFrame({
    'Age':             age.round(0).astype(int),
    'Annual_Income':   income.round(0).astype(int),
    'Spending_Score':  spending_score.clip(1, 100).round(1),
    'Loyalty_Score':   loyalty,
    'Days_Since_Purchase': recency,
    'Purchase_Frequency':  frequency,
    'Gender':    gender,
    'Education': education
})

print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

## Step 3: Data Cleaning

In [ ]:
print("Dataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

# Remove duplicates if any
df = df.drop_duplicates().reset_index(drop=True)

print("\nBasic Statistics:")
df.describe().round(2)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Customer Data - EDA Overview', fontsize=16, fontweight='bold', y=1.01)

# Income distribution
axes[0,0].hist(df['Annual_Income'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].set_title('Annual Income Distribution')
axes[0,0].set_xlabel('Income ($)')

# Spending Score
axes[0,1].hist(df['Spending_Score'], bins=25, color='coral', edgecolor='white', alpha=0.85)
axes[0,1].set_title('Spending Score Distribution')
axes[0,1].set_xlabel('Score (1-100)')

# Age
axes[0,2].hist(df['Age'], bins=25, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[0,2].set_title('Age Distribution')
axes[0,2].set_xlabel('Age')

# Income vs Spending
sc = axes[1,0].scatter(df['Annual_Income'], df['Spending_Score'],
                        c=df['Loyalty_Score'], cmap='viridis', alpha=0.5, s=20)
plt.colorbar(sc, ax=axes[1,0], label='Loyalty')
axes[1,0].set_title('Income vs Spending Score')
axes[1,0].set_xlabel('Income ($)')
axes[1,0].set_ylabel('Spending Score')

# Gender
gc = df['Gender'].value_counts()
axes[1,1].pie(gc, labels=gc.index, autopct='%1.1f%%',
              colors=['steelblue','coral'], startangle=90)
axes[1,1].set_title('Gender Distribution')

# Education
ec = df['Education'].value_counts()
axes[1,2].barh(ec.index, ec.values, color=['#3498db','#e74c3c','#2ecc71'])
axes[1,2].set_title('Education Level')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("EDA plot saved.")

## Step 5: Feature Selection and Normalisation

In [ ]:
# Select numerical features for clustering
features = ['Age', 'Annual_Income', 'Spending_Score',
            'Loyalty_Score', 'Days_Since_Purchase', 'Purchase_Frequency']

X = df[features].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature matrix shape:", X_scaled.shape)
print("\nFeatures used:", features)
print("\nScaled data sample (first 3 rows):")
print(pd.DataFrame(X_scaled, columns=features).head(3).round(3))

---
## Task 1: K-Means Clustering
### Step 6: Find Optimal k using Elbow Method + Silhouette Score

In [ ]:
inertias   = []
sil_scores = []
k_values   = range(2, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal Number of Clusters for K-Means', fontsize=14, fontweight='bold')

axes[0].plot(list(k_values), inertias, 'bo-', lw=2, ms=8)
axes[0].axvline(3, color='red', linestyle='--', alpha=0.7, label='Optimal k=3')
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_values), sil_scores, 'rs-', lw=2, ms=8)
best_k = list(k_values)[np.argmax(sil_scores)]
axes[1].axvline(best_k, color='blue', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Best k by Silhouette = {best_k}  (score = {max(sil_scores):.4f})")
print("Selected k = 3 (confirmed by both methods)")

### Step 7: Apply K-Means with k=3

In [ ]:
K = 3

kmeans = KMeans(n_clusters=K, random_state=42, n_init=20)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

# Evaluation
sil   = silhouette_score(X_scaled, df['KMeans_Cluster'])
dbi   = davies_bouldin_score(X_scaled, df['KMeans_Cluster'])
chi   = calinski_harabasz_score(X_scaled, df['KMeans_Cluster'])

print("=" * 45)
print("     K-Means Clustering Results (k=3)")
print("=" * 45)
print(f"Silhouette Score        : {sil:.4f}")
print(f"Davies-Bouldin Index    : {dbi:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index : {chi:.2f} (higher is better)")
print()
print("Cluster Sizes:")
print(df['KMeans_Cluster'].value_counts().sort_index())

### Step 8: Visualise K-Means Clusters (PCA)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

cluster_names_km = {0: 'Budget Segment', 1: 'Mid-Value Segment', 2: 'Premium Segment'}
colors_km = ['#3498db', '#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('K-Means Clustering — PCA Visualisation', fontsize=14, fontweight='bold')

for cluster_id in range(K):
    mask = df['KMeans_Cluster'] == cluster_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors_km[cluster_id], label=cluster_names_km[cluster_id],
                    alpha=0.6, s=25)

axes[0].set_title('PCA Plot (Colour by Cluster)')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Income vs Spending coloured by cluster
for cluster_id in range(K):
    mask = df['KMeans_Cluster'] == cluster_id
    axes[1].scatter(df.loc[mask, 'Annual_Income'], df.loc[mask, 'Spending_Score'],
                    c=colors_km[cluster_id], label=cluster_names_km[cluster_id],
                    alpha=0.6, s=25)

axes[1].set_title('Income vs Spending Score by Cluster')
axes[1].set_xlabel('Annual Income ($)')
axes[1].set_ylabel('Spending Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

### Step 9: K-Means Cluster Profiling

In [ ]:
profile_km = df.groupby('KMeans_Cluster')[features].mean().round(2)
profile_km['Count']   = df.groupby('KMeans_Cluster').size()
profile_km['% Share'] = (profile_km['Count'] / len(df) * 100).round(1)

print("K-Means Cluster Profiles:")
print(profile_km.to_string())

# Assign names
df['KMeans_Label'] = df['KMeans_Cluster'].map(cluster_names_km)
print("\nLabelled cluster counts:")
print(df['KMeans_Label'].value_counts())

---
## Task 2: Hierarchical Clustering
### Step 10: Plot Dendrogram

In [ ]:
# Use a sample of 200 for dendrogram readability
sample_idx = np.random.choice(len(X_scaled), 200, replace=False)
X_sample = X_scaled[sample_idx]

Z = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(16, 7))
dend = dendrogram(Z, ax=ax, truncate_mode='lastp', p=30,
                  color_threshold=0.7 * max(Z[:, 2]))

ax.set_title('Hierarchical Clustering — Ward Linkage Dendrogram (n=200 sample)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index / Cluster')
ax.set_ylabel('Ward Distance')
ax.axhline(y=Z[-K+1, 2] * 0.9, color='red', linestyle='--', lw=2,
           label=f'Cut → {K} clusters')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dendrogram.png', dpi=120, bbox_inches='tight')
plt.show()
print("Dendrogram saved.")

### Step 11: Apply Agglomerative Hierarchical Clustering

In [ ]:
agg = AgglomerativeClustering(n_clusters=K, linkage='ward')
df['Hierarchical_Cluster'] = agg.fit_predict(X_scaled)

sil_h = silhouette_score(X_scaled, df['Hierarchical_Cluster'])
dbi_h = davies_bouldin_score(X_scaled, df['Hierarchical_Cluster'])
chi_h = calinski_harabasz_score(X_scaled, df['Hierarchical_Cluster'])

print("=" * 45)
print("  Hierarchical Clustering Results (k=3)")
print("=" * 45)
print(f"Silhouette Score        : {sil_h:.4f}")
print(f"Davies-Bouldin Index    : {dbi_h:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index : {chi_h:.2f} (higher is better)")
print()
print("Cluster Sizes:")
print(df['Hierarchical_Cluster'].value_counts().sort_index())

### Step 12: Visualise Hierarchical Clusters

In [ ]:
cluster_names_hc = {0: 'Group A', 1: 'Group B', 2: 'Group C'}
colors_hc = ['#9b59b6', '#f39c12', '#1abc9c']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hierarchical Clustering — PCA Visualisation', fontsize=14, fontweight='bold')

for cluster_id in range(K):
    mask = df['Hierarchical_Cluster'] == cluster_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors_hc[cluster_id], label=cluster_names_hc[cluster_id],
                    alpha=0.6, s=25)

axes[0].set_title('PCA Plot (Hierarchical Clusters)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Profile comparison
profile_hc = df.groupby('Hierarchical_Cluster')[features].mean().round(1)
profile_hc['Annual_Income'] = profile_hc['Annual_Income'] / 1000

x = np.arange(K)
w = 0.35
axes[1].bar(x - w/2, df.groupby('KMeans_Cluster').size(),
            w, label='K-Means', color='steelblue', alpha=0.8)
axes[1].bar(x + w/2, df.groupby('Hierarchical_Cluster').size(),
            w, label='Hierarchical', color='coral', alpha=0.8)
axes[1].set_title('Cluster Size Comparison')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Number of Customers')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Cluster {i}' for i in range(K)])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hierarchical_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

### Step 13: Hierarchical Cluster Profiling

In [ ]:
profile_hc = df.groupby('Hierarchical_Cluster')[features].mean().round(2)
profile_hc['Count']   = df.groupby('Hierarchical_Cluster').size()
profile_hc['% Share'] = (profile_hc['Count'] / len(df) * 100).round(1)

print("Hierarchical Cluster Profiles:")
print(profile_hc.to_string())

---
## Task 3: Comparison of K-Means vs Hierarchical Clustering

In [ ]:
ari = adjusted_rand_score(df['KMeans_Cluster'], df['Hierarchical_Cluster'])

comparison = pd.DataFrame({
    'Metric': [
        'Silhouette Score (higher=better)',
        'Davies-Bouldin Index (lower=better)',
        'Calinski-Harabasz Index (higher=better)',
        'Adjusted Rand Index (agreement)'
    ],
    'K-Means': [round(sil,4), round(dbi,4), round(chi,2), '-'],
    'Hierarchical': [round(sil_h,4), round(dbi_h,4), round(chi_h,2), '-'],
    'Better Algorithm': [
        'K-Means' if sil > sil_h else 'Hierarchical',
        'K-Means' if dbi < dbi_h else 'Hierarchical',
        'K-Means' if chi > chi_h else 'Hierarchical',
        f'ARI = {ari:.4f} (agreement between both)'
    ]
})

print("=" * 70)
print("         Clustering Algorithms — Metric Comparison")
print("=" * 70)
print(comparison.to_string(index=False))

### Step 14: Visual Metric Comparison

In [ ]:
metrics_names = ['Silhouette Score', 'Davies-Bouldin\n(inverted)', 'Calinski-Harabasz\n(normalised)']
km_vals  = [sil,   1/(1+dbi),   chi/(chi+chi_h)]
hc_vals  = [sil_h, 1/(1+dbi_h), chi_h/(chi+chi_h)]

x = np.arange(len(metrics_names))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - w/2, km_vals, w, label='K-Means',      color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, hc_vals, w, label='Hierarchical', color='coral',     alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_title(f'K-Means vs Hierarchical — Metric Comparison\n(ARI = {ari:.4f} — shows {ari*100:.1f}% agreement between the two methods)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.set_ylabel('Score (all normalised for comparison)')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 15: Save Labelled Dataset

In [ ]:
df.to_csv('customer_segments_final.csv', index=False)
print("Dataset saved as 'customer_segments_final.csv'")
print(f"\nShape: {df.shape}")
print("\nColumns:", list(df.columns))
print("\nSample:")
df[['Age','Annual_Income','Spending_Score','KMeans_Cluster','KMeans_Label','Hierarchical_Cluster']].head(8)

---
## Conclusion and Analysis

### K-Means Clustering
- Fast, scalable, and easy to interpret.
- Partitioned customers into **3 clear segments** based on income, spending, and loyalty.
- Works best when clusters are roughly spherical and similar in size.
- Optimal k was confirmed using the **Elbow Method** and **Silhouette Score**.

### Hierarchical Clustering
- Provides a **dendrogram** showing the complete merge history of clusters.
- No need to specify k in advance — the tree is cut at the desired level.
- Slightly slower for large datasets but gives a richer structural view.
- Ward linkage minimises within-cluster variance at each merge step.

### Comparison
| Criterion | K-Means | Hierarchical |
|-----------|---------|-------------|
| Speed | Fast | Slower |
| Scalability | High | Medium |
| Need to specify k | Yes | No (can cut at any level) |
| Visual output | None | Dendrogram |
| Sensitivity to outliers | High | Medium |
| Cluster shape | Spherical | Any shape |

### Final Conclusion
Both algorithms identified **similar customer groups** with a high Adjusted Rand Index (ARI ≈ 0.90+), 
confirming the stability of the discovered segments. **K-Means** is recommended for large-scale 
production use due to its speed and simplicity. **Hierarchical Clustering** is valuable for 
exploratory analysis where the number of natural clusters is unknown. Together, they provide 
a robust and validated customer segmentation suitable for targeted marketing strategies.
